In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

C:\Users\Bikram\anaconda3\envs\earning_call-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
model_path = "sentiment/final"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model = model.to(device)
model.eval()

print("Model loaded:", device)
print("Labels:", model.config.id2label)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1873.36it/s]


Model loaded: cuda
Labels: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [10]:
def predict_sentiment(texts, batch_size=32):
    all_scores = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        
        encodings = tokenizer(
            batch, truncation=True, padding=True,
            max_length=128, return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            logits = model(**encodings).logits
        
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        
        for p in probs:
            all_scores.append({
                "negative": float(p[0]),
                "neutral": float(p[1]),
                "positive": float(p[2])
            })
    
    return all_scores

In [8]:
import re

In [9]:
def split_into_chunks(text, max_words=300):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sentence in sentences:
        sentence_word_count = len(sentence.split())
        
        if current_word_count + sentence_word_count > max_words and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_word_count = 0
        
        current_chunk.append(sentence)
        current_word_count += sentence_word_count
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

In [6]:
def score_text(text, max_words=300):
    if len(text.split()) <= max_words:
        chunks = [text]
    else:
        chunks = split_into_chunks(text, max_words=max_words)
    
    chunk_scores = predict_sentiment(chunks)
    
    avg_negative = np.mean([s["negative"] for s in chunk_scores])
    avg_neutral = np.mean([s["neutral"] for s in chunk_scores])
    avg_positive = np.mean([s["positive"] for s in chunk_scores])
    
    return {
        "negative": float(avg_negative),
        "neutral": float(avg_neutral),
        "positive": float(avg_positive),
        "num_chunks": len(chunks)
    }

In [4]:
def score_quarter_weighted(prepared_turns, qa_pairs, prepared_weight=0.3, qa_weight=0.7):
    prepared_texts = [t["text"] for t in prepared_turns if len(t["text"].split()) > 5]
    answer_texts = [p["answer"] for p in qa_pairs if len(p["answer"].split()) > 5]
    
    result = {
        "prepared_score": None,
        "qa_score": None,
        "final_score": None,
        "prepared_count": len(prepared_texts),
        "qa_count": len(answer_texts)
    }
    
    prepared_avg = None
    qa_avg = None
    
    if prepared_texts:
        prepared_scores = [score_text(t) for t in prepared_texts]
        prepared_avg = {
            "negative": np.mean([s["negative"] for s in prepared_scores]),
            "neutral": np.mean([s["neutral"] for s in prepared_scores]),
            "positive": np.mean([s["positive"] for s in prepared_scores])
        }
        result["prepared_score"] = prepared_avg
    
    if answer_texts:
        qa_scores = [score_text(t) for t in answer_texts]
        qa_avg = {
            "negative": np.mean([s["negative"] for s in qa_scores]),
            "neutral": np.mean([s["neutral"] for s in qa_scores]),
            "positive": np.mean([s["positive"] for s in qa_scores])
        }
        result["qa_score"] = qa_avg
    
    if prepared_avg and qa_avg:
        final = {
            "negative": prepared_weight * prepared_avg["negative"] + qa_weight * qa_avg["negative"],
            "neutral": prepared_weight * prepared_avg["neutral"] + qa_weight * qa_avg["neutral"],
            "positive": prepared_weight * prepared_avg["positive"] + qa_weight * qa_avg["positive"]
        }
    elif qa_avg:
        final = qa_avg
    elif prepared_avg:
        final = prepared_avg
    else:
        final = {"negative": 0, "neutral": 1, "positive": 0}
    
    result["final_score"] = final
    return result

In [10]:
PROCESSED_DIR = Path("../data/processed")
processed_files = list(PROCESSED_DIR.glob("*.json"))

print("Total transcripts to process:", len(processed_files))
print()

quarterly_sentiment = []

for filepath in processed_files:
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)
    
    prepared_turns = data.get("prepared_remarks", [])
    qa_pairs = data.get("qa_pairs", [])
    
    result = score_quarter_weighted(
        prepared_turns,
        qa_pairs,
        prepared_weight=0.3,
        qa_weight=0.7
    )
    
    final = result["final_score"]
    
    quarterly_sentiment.append({
        "symbol": data["symbol"],
        "quarter": data["quarter"],
        "year": data["year"],
        "positive": round(float(final["positive"]), 4),
        "negative": round(float(final["negative"]), 4),
        "neutral": round(float(final["neutral"]), 4),
        "prepared_positive": round(float(result["prepared_score"]["positive"]), 4) if result["prepared_score"] else None,
        "qa_positive": round(float(result["qa_score"]["positive"]), 4) if result["qa_score"] else None,
        "prepared_count": result["prepared_count"],
        "qa_count": result["qa_count"]
    })
    
    print(data["symbol"], "Q" + str(data["quarter"]), data["year"],
          "-> pos:", round(float(final["positive"]), 3),
          "neg:", round(float(final["negative"]), 3),
          "neu:", round(float(final["neutral"]), 3))

print()
print("Total quarters processed:", len(quarterly_sentiment))

Total transcripts to process: 160

AAPL Q1 2021 -> pos: 0.283 neg: 0.0 neu: 0.716
AAPL Q1 2022 -> pos: 0.395 neg: 0.002 neu: 0.603
AAPL Q1 2023 -> pos: 0.282 neg: 0.065 neu: 0.653
AAPL Q1 2024 -> pos: 0.259 neg: 0.036 neu: 0.705
AAPL Q2 2021 -> pos: 0.373 neg: 0.01 neu: 0.617
AAPL Q2 2022 -> pos: 0.132 neg: 0.007 neu: 0.861
AAPL Q2 2023 -> pos: 0.352 neg: 0.076 neu: 0.572
AAPL Q2 2024 -> pos: 0.287 neg: 0.036 neu: 0.676
AAPL Q3 2021 -> pos: 0.36 neg: 0.01 neu: 0.629
AAPL Q3 2022 -> pos: 0.202 neg: 0.068 neu: 0.73
AAPL Q3 2023 -> pos: 0.375 neg: 0.059 neu: 0.566
AAPL Q3 2024 -> pos: 0.238 neg: 0.0 neu: 0.761
AAPL Q4 2021 -> pos: 0.168 neg: 0.037 neu: 0.795
AAPL Q4 2022 -> pos: 0.151 neg: 0.057 neu: 0.792
AAPL Q4 2023 -> pos: 0.155 neg: 0.043 neu: 0.801
AAPL Q4 2024 -> pos: 0.246 neg: 0.005 neu: 0.749
AMZN Q1 2021 -> pos: 0.424 neg: 0.0 neu: 0.575
AMZN Q1 2022 -> pos: 0.185 neg: 0.031 neu: 0.784
AMZN Q1 2023 -> pos: 0.147 neg: 0.021 neu: 0.832
AMZN Q1 2024 -> pos: 0.401 neg: 0.0 neu: 0.5

In [11]:
import pandas as pd
print('ok')
df = pd.DataFrame(quarterly_sentiment)

print("Shape:", df.shape)
print()
print("Per-company average positive sentiment:")
print(df.groupby("symbol")["positive"].mean().round(3).sort_values(ascending=False))
print()
print("Per-company positive sentiment std deviation (variation):")
print(df.groupby("symbol")["positive"].std().round(3).sort_values(ascending=False))
print()
print("Overall range:")
print("  Min positive:", df["positive"].min())
print("  Max positive:", df["positive"].max())
print("  Std across all quarters:", round(df["positive"].std(), 4))

ok
Shape: (160, 10)

Per-company average positive sentiment:
symbol
AON     0.357
WTW     0.352
GS      0.296
AMZN    0.276
AAPL    0.266
MET     0.259
MS      0.196
MSFT    0.154
JPM     0.104
TSLA    0.070
Name: positive, dtype: float64

Per-company positive sentiment std deviation (variation):
symbol
WTW     0.109
AMZN    0.094
MET     0.093
AON     0.090
AAPL    0.088
MSFT    0.067
MS      0.065
GS      0.056
TSLA    0.055
JPM     0.051
Name: positive, dtype: float64

Overall range:
  Min positive: 0.0072
  Max positive: 0.5488
  Std across all quarters: 0.1215


In [12]:
output_path = Path("../data/quarterly_sentiment.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(quarterly_sentiment, f, indent=2)

print("Saved", len(quarterly_sentiment), "quarterly sentiment scores")
print("Path:", output_path)
print()
print("Sample entry:")
print(quarterly_sentiment[0])

Saved 160 quarterly sentiment scores
Path: ..\data\quarterly_sentiment.json

Sample entry:
{'symbol': 'AAPL', 'quarter': 1, 'year': 2021, 'positive': 0.2832, 'negative': 0.0003, 'neutral': 0.7164, 'prepared_positive': 0.21, 'qa_positive': 0.3146, 'prepared_count': 31, 'qa_count': 8}


In [3]:
from pathlib import Path
import json

PROCESSED_DIR  = Path("../data/processed")
OUTPUT_PATH    = Path("../data/quarterly_sentiment.json")

# Load existing scores so we don't redo them
with open(OUTPUT_PATH) as f:
    existing_scores = json.load(f)

# Build set of already-scored quarters
already_scored = set()
for entry in existing_scores:
    key = f"{entry['symbol']}_{entry['quarter']}_{entry['year']}"
    already_scored.add(key)

print("Already scored quarters:", len(already_scored))

# Get all processed files
all_processed = list(PROCESSED_DIR.glob("*.json"))

# Filter to only unscored files
new_files = []
for filepath in all_processed:
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)
    key = f"{data['symbol']}_{data['quarter']}_{data['year']}"
    if key not in already_scored:
        new_files.append(filepath)

print("New files to score:", len(new_files))

Already scored quarters: 160
New files to score: 432


In [12]:
new_quarterly_sentiment = []

for filepath in new_files:
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    prepared_turns = data.get("prepared_remarks", [])
    qa_pairs       = data.get("qa_pairs", [])

    result = score_quarter_weighted(
        prepared_turns,
        qa_pairs,
        prepared_weight=0.3,
        qa_weight=0.7
    )

    final = result["final_score"]

    new_quarterly_sentiment.append({
        "symbol": data["symbol"],
        "quarter": data["quarter"],
        "year": data["year"],
        "positive": round(float(final["positive"]), 4),
        "negative": round(float(final["negative"]), 4),
        "neutral": round(float(final["neutral"]), 4),
        "prepared_positive": round(float(result["prepared_score"]["positive"]), 4) if result["prepared_score"] else None,
        "qa_positive": round(float(result["qa_score"]["positive"]), 4) if result["qa_score"] else None,
        "prepared_count": result["prepared_count"],
        "qa_count": result["qa_count"]
    })

    print(data["symbol"], "Q" + str(data["quarter"]), data["year"],
          "-> pos:", round(float(final["positive"]), 3),
          "neg:", round(float(final["negative"]), 3),
          "neu:", round(float(final["neutral"]), 3))

print()
print("New quarters scored:", len(new_quarterly_sentiment))

AFL Q1 2021 -> pos: 0.215 neg: 0.104 neu: 0.681
AFL Q1 2022 -> pos: 0.179 neg: 0.111 neu: 0.709
AFL Q1 2023 -> pos: 0.144 neg: 0.026 neu: 0.83
AFL Q1 2024 -> pos: 0.301 neg: 0.0 neu: 0.699
AFL Q2 2021 -> pos: 0.192 neg: 0.068 neu: 0.74
AFL Q2 2022 -> pos: 0.224 neg: 0.022 neu: 0.754
AFL Q2 2023 -> pos: 0.258 neg: 0.017 neu: 0.726
AFL Q2 2024 -> pos: 0.366 neg: 0.001 neu: 0.633
AFL Q3 2021 -> pos: 0.155 neg: 0.071 neu: 0.773
AFL Q3 2022 -> pos: 0.196 neg: 0.002 neu: 0.802
AFL Q3 2023 -> pos: 0.267 neg: 0.026 neu: 0.707
AFL Q3 2024 -> pos: 0.32 neg: 0.059 neu: 0.622
AFL Q4 2021 -> pos: 0.168 neg: 0.003 neu: 0.829
AFL Q4 2022 -> pos: 0.153 neg: 0.069 neu: 0.778
AFL Q4 2023 -> pos: 0.297 neg: 0.022 neu: 0.681
AFL Q4 2024 -> pos: 0.339 neg: 0.059 neu: 0.601
ALL Q1 2021 -> pos: 0.237 neg: 0.017 neu: 0.746
ALL Q1 2022 -> pos: 0.224 neg: 0.042 neu: 0.734
ALL Q1 2023 -> pos: 0.215 neg: 0.019 neu: 0.766
ALL Q1 2024 -> pos: 0.251 neg: 0.0 neu: 0.748
ALL Q2 2021 -> pos: 0.184 neg: 0.0 neu: 0.816
A

In [13]:
all_quarterly_sentiment = existing_scores + new_quarterly_sentiment

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_quarterly_sentiment, f, indent=2)

print("Total quarters now:", len(all_quarterly_sentiment))
print("Saved to:", OUTPUT_PATH)

Total quarters now: 592
Saved to: ..\data\quarterly_sentiment.json
